# ACDADA — Notebook 06: Deception RL Agent

**PPO & DQN Agents for Cyber Deception Strategy Learning**

This notebook implements:
1. Custom DQN agent (from scratch in PyTorch)
2. PPO agent via Stable-Baselines3
3. Training loop with curriculum learning
4. Agent evaluation & comparison with baselines
5. Model export for the orchestration pipeline

In [ ]:
# ============================================================
# DATASET / REFERENCE LINKS
# ============================================================
#
# CybORG (Cyber Operations Research Gym) — reference:
#   https://github.com/cage-challenge/CybORG
#
# Stable-Baselines3 documentation:
#   https://stable-baselines3.readthedocs.io/
#
# This notebook trains RL agents on the CyberDeceptionEnv
# defined in Notebook 05.
# ============================================================

## 1. Imports & Configuration

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import deque, namedtuple
from typing import Dict, List, Tuple, Optional
import json
import copy
import random
import warnings
import gymnasium as gym
from gymnasium import spaces

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = Path('../models/deception_agent')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 2. Re-import Environment from Notebook 05

We redefine the environment inline so this notebook is self-contained.

In [ ]:
from enum import IntEnum
from dataclasses import dataclass, field

class ServiceType(IntEnum):
    HTTP = 0; SSH = 1; FTP = 2; DNS = 3; DATABASE = 4; SMTP = 5

class AttackPhase(IntEnum):
    RECONNAISSANCE = 0; SCANNING = 1; EXPLOITATION = 2
    LATERAL_MOVEMENT = 3; EXFILTRATION = 4

class HostStatus(IntEnum):
    SECURE = 0; PROBED = 1; COMPROMISED = 2

@dataclass
class Host:
    host_id: int; name: str; is_honeypot: bool = False
    is_critical: bool = False; services: List[int] = field(default_factory=list)
    vulnerability_score: float = 0.5; status: int = HostStatus.SECURE; value: float = 1.0

def create_default_network(n_real=8, n_hp=4):
    hosts, hid = {}, 0
    configs = [
        ('web-server-01', [0,1], 0.6, False, 2.0), ('dns-server-01', [3,1], 0.4, False, 1.5),
        ('db-server-01', [4,1], 0.3, True, 5.0), ('mail-server-01', [5,0], 0.5, False, 2.0),
    ]
    for name, svcs, vuln, crit, val in configs:
        hosts[hid] = Host(hid, name, services=svcs, vulnerability_score=vuln, is_critical=crit, value=val); hid+=1
    for i in range(n_real - 4):
        hosts[hid] = Host(hid, f'workstation-{i+1:02d}', services=[0,1,2],
                          vulnerability_score=np.random.uniform(0.3,0.7)); hid+=1
    sp = [[0,1],[4,1],[2,0],[5,3]]
    for i in range(n_hp):
        hosts[hid] = Host(hid, f'honeypot-{i+1:02d}', is_honeypot=True,
                          services=sp[i%4], vulnerability_score=0.8, value=0.0); hid+=1
    return hosts

class SimulatedAttacker:
    def __init__(self, skill=0.5, persistence=0.7):
        self.skill=skill; self.persistence=persistence; self.reset()
    def reset(self):
        self.phase=0; self.position=-1; self.discovered=set(); self.compromised=set()
        self.target=None; self.time_in_net=0; self.exfil=0.0
    def step(self, hosts, active_hp):
        self.time_in_net += 1
        if self.phase==0: return self._recon(hosts, active_hp)
        if self.phase==1: return self._scan(hosts)
        if self.phase==2: return self._exploit(hosts)
        if self.phase==3: return self._lateral(hosts)
        if self.phase==4: return self._exfil(hosts)
        return {'phase':-1,'target':None,'success':False,'noise_level':0}
    def _recon(self, hosts, active_hp):
        vis = [h for h in hosts if not hosts[h].is_honeypot or h in active_hp]
        n = min(np.random.randint(2,5), len(vis))
        found = set(np.random.choice(vis, n, replace=False))
        self.discovered.update(found)
        if len(self.discovered)>=3: self.phase=1
        return {'phase':0,'target':list(found),'success':True,'noise_level':0.2}
    def _scan(self, hosts):
        cands = [(h,hosts[h].vulnerability_score) for h in self.discovered if h not in self.compromised and h in hosts]
        if not cands: self.phase=0; return {'phase':1,'target':None,'success':False,'noise_level':0.1}
        hids,vs = zip(*cands); vs=np.array(vs); ps=vs/vs.sum()
        self.target = np.random.choice(hids, p=ps); self.phase=2
        return {'phase':1,'target':self.target,'success':True,'noise_level':0.4}
    def _exploit(self, hosts):
        if self.target is None or self.target not in hosts:
            self.phase=1; return {'phase':2,'target':None,'success':False,'noise_level':0.3}
        t=hosts[self.target]; p=self.skill*t.vulnerability_score; s=np.random.random()<p
        if s:
            self.compromised.add(self.target); self.position=self.target; t.status=2
            self.phase = 4 if t.is_critical else 3
        else:
            t.status=max(t.status,1)
            if np.random.random()>=self.persistence: self.phase=1
        return {'phase':2,'target':self.target,'success':s,'noise_level':0.7}
    def _lateral(self, hosts):
        new = [h for h in self.discovered if h not in self.compromised and h in hosts]
        if not new: self.phase=0; return {'phase':3,'target':None,'success':False,'noise_level':0.5}
        crit = [h for h in new if hosts[h].is_critical]
        if crit: self.target=np.random.choice(crit)
        else:
            vs=np.array([hosts[h].vulnerability_score for h in new]); ps=vs/vs.sum()
            self.target=np.random.choice(new, p=ps)
        self.phase=2
        return {'phase':3,'target':self.target,'success':True,'noise_level':0.5}
    def _exfil(self, hosts):
        cc = [h for h in self.compromised if h in hosts and hosts[h].is_critical]
        if not cc: self.phase=3; return {'phase':4,'target':None,'success':False,'noise_level':0.3}
        amt = hosts[cc[0]].value*self.skill; self.exfil+=amt
        return {'phase':4,'target':cc[0],'success':True,'noise_level':0.8,'data_exfiltrated':amt}

class CyberDeceptionEnv(gym.Env):
    metadata = {'render_modes': ['human','ansi']}
    def __init__(self, n_real=8, n_hp=4, max_steps=100, atk_skill=0.5, render_mode=None):
        super().__init__()
        self.n_real=n_real; self.n_hp=n_hp; self.n_total=n_real+n_hp
        self.max_steps=max_steps; self.atk_skill=atk_skill; self.render_mode=render_mode
        self.hf=7; self.gf=11; obs_sz=self.n_total*self.hf+self.gf
        self.observation_space=spaces.Box(0,1,(obs_sz,),np.float32)
        self.action_space=spaces.Discrete(9)
        self.hosts={}; self.attacker=None; self.active_hp=[]; self.step_n=0
        self.mon=0.0; self.blocked=False; self.hp_int=0; self.real_comp=0; self.det_time=-1
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.hosts=create_default_network(self.n_real,self.n_hp)
        self.attacker=SimulatedAttacker(self.atk_skill); self.active_hp=[]; self.step_n=0
        self.mon=0.3; self.blocked=False; self.hp_int=0; self.real_comp=0; self.det_time=-1
        return self._obs(), {}
    def _obs(self):
        o=[]
        for h in range(self.n_total):
            H=self.hosts[h]
            o.extend([H.status/2, 1.0 if(H.is_honeypot and h in self.active_hp) else 0,
                      H.vulnerability_score, 1.0 if H.is_critical else 0, len(H.services)/6,
                      1.0 if H.status>=1 else 0, 1.0 if H.status>=2 else 0])
        det=self._detected(); ph=[0]*5
        if det: ph[int(self.attacker.phase)]=1.0
        nc=sum(1 for h in self.hosts.values() if h.status==2)
        np_=sum(1 for h in self.hosts.values() if h.status>=1)
        o.extend([1.0 if det else 0]+ph+[self.step_n/self.max_steps, nc/self.n_total,
                  len(self.active_hp)/max(self.n_hp,1), np_/self.n_total, self.mon])
        return np.array(o, dtype=np.float32)
    def _detected(self):
        if self.blocked: return True
        for h in self.attacker.compromised:
            if h in self.active_hp: return True
        nf={0:0.1,1:0.3,2:0.5,3:0.4,4:0.6}.get(self.attacker.phase,0.1)
        return np.random.random()<(self.mon*0.3+nf*self.mon)
    def step(self, action):
        self.step_n+=1; r=0.0; info={'action_name':'','attacker_action':{}}
        hpids=[h for h,H in self.hosts.items() if H.is_honeypot]
        if action==0: info['action_name']='observe'
        elif 1<=action<=4:
            idx=action-1
            if idx<len(hpids):
                hid=hpids[idx]
                if hid not in self.active_hp: self.active_hp.append(hid); r-=0.05
                else: r-=0.01
        elif action==5: self.active_hp=[]
        elif action==6:
            if self._detected() and self.active_hp:
                self.attacker.target=np.random.choice(self.active_hp); self.attacker.phase=2; r+=0.3
            else: r-=0.02
        elif action==7: self.mon=min(1.0,self.mon+0.15); r-=0.03
        elif action==8:
            if self._detected(): self.blocked=True; r+=1.0
            else: r-=0.5
        aa={'phase':-1,'target':None,'success':False}
        if not self.blocked:
            aa=self.attacker.step(self.hosts, self.active_hp); info['attacker_action']=aa
            tgt=aa.get('target')
            if tgt is not None:
                if isinstance(tgt,(list,np.ndarray)):
                    for t in tgt:
                        if t in self.active_hp: self.hp_int+=1; r+=0.5
                elif tgt in self.active_hp:
                    self.hp_int+=1; r+=0.5
                    if aa.get('success'): r+=1.0
            if aa.get('success') and tgt is not None and not isinstance(tgt,(list,np.ndarray)):
                if tgt in self.hosts and not self.hosts[tgt].is_honeypot:
                    self.real_comp+=1; r-=self.hosts[tgt].value*0.5
                    if self.hosts[tgt].is_critical: r-=2.0
            if aa.get('data_exfiltrated',0)>0: r-=aa['data_exfiltrated']*2.0
        if self.det_time==-1 and self._detected():
            self.det_time=self.step_n; r+=max(0,1.0-self.step_n/self.max_steps)
        done=False; trunc=False
        if self.blocked: done=True
        if self.attacker.exfil>5.0: done=True; r-=5.0
        if self.step_n>=self.max_steps: trunc=True; r-=1.0 if not self.blocked else 0
        info.update(honeypot_interactions=self.hp_int, real_compromises=self.real_comp,
                    detection_time=self.det_time, attacker_time_in_network=self.attacker.time_in_net,
                    exfiltrated_data=self.attacker.exfil)
        return self._obs(), r, done, trunc, info

# Verify env
env = CyberDeceptionEnv()
obs, _ = env.reset()
print(f'Env ready. Obs shape: {obs.shape}, Actions: {env.action_space.n}')

---
## 3. Deep Q-Network (DQN) — From Scratch

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward', 'done'))


class ReplayBuffer:
    """Experience replay buffer for DQN."""
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, *args):
        self.buffer.append(Transition(*args))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))
    
    def __len__(self):
        return len(self.buffer)


class DuelingDQN(nn.Module):
    """
    Dueling DQN architecture.
    Separates state-value V(s) and advantage A(s,a) streams.
    Q(s,a) = V(s) + A(s,a) - mean(A(s,.))
    """
    def __init__(self, obs_dim, n_actions, hidden=256):
        super().__init__()
        
        # Shared feature extractor
        self.features = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.LayerNorm(hidden),
        )
        
        # Value stream
        self.value_stream = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        
        # Advantage stream
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions),
        )
    
    def forward(self, x):
        features = self.features(x)
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        # Combine: Q = V + (A - mean(A))
        q_values = value + advantage - advantage.mean(dim=-1, keepdim=True)
        return q_values


print(f'DuelingDQN params: {sum(p.numel() for p in DuelingDQN(95, 9).parameters()):,}')

In [ ]:
class DQNAgent:
    """
    DQN Agent with:
    - Dueling architecture
    - Double DQN (use online net for action selection, target net for value)
    - Experience replay
    - Epsilon-greedy exploration with decay
    """
    
    def __init__(self, obs_dim, n_actions, lr=3e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay=5000,
                 batch_size=64, target_update=500, buffer_size=50000):
        self.obs_dim = obs_dim
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        
        # Epsilon schedule
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        self.steps_done = 0
        
        # Networks
        self.policy_net = DuelingDQN(obs_dim, n_actions).to(DEVICE)
        self.target_net = DuelingDQN(obs_dim, n_actions).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        self.optimizer = optim.AdamW(self.policy_net.parameters(), lr=lr, weight_decay=1e-5)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=2000, gamma=0.95)
        self.memory = ReplayBuffer(buffer_size)
        
        # Tracking
        self.train_losses = []
    
    def get_epsilon(self):
        return self.eps_end + (self.eps_start - self.eps_end) * \
               np.exp(-self.steps_done / self.eps_decay)
    
    def predict(self, state, deterministic=False):
        """Select action using epsilon-greedy."""
        eps = 0.0 if deterministic else self.get_epsilon()
        
        if random.random() < eps:
            return random.randrange(self.n_actions), None
        
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            q_values = self.policy_net(state_t)
            return q_values.argmax(dim=1).item(), None
    
    def store(self, state, action, next_state, reward, done):
        self.memory.push(state, action, next_state, reward, done)
    
    def train_step(self):
        """Perform one gradient update."""
        if len(self.memory) < self.batch_size:
            return None
        
        batch = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(np.array(batch.state)).to(DEVICE)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(DEVICE)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(DEVICE)
        rewards = torch.FloatTensor(batch.reward).unsqueeze(1).to(DEVICE)
        dones = torch.FloatTensor(batch.done).unsqueeze(1).to(DEVICE)
        
        # Current Q values
        current_q = self.policy_net(states).gather(1, actions)
        
        # Double DQN: action from policy net, value from target
        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(1, keepdim=True)
            next_q = self.target_net(next_states).gather(1, next_actions)
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # Huber loss
        loss = F.smooth_l1_loss(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()
        self.scheduler.step()
        
        self.steps_done += 1
        
        # Update target network
        if self.steps_done % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        self.train_losses.append(loss.item())
        return loss.item()

print('DQNAgent defined.')

---
## 4. DQN Training

In [ ]:
def train_dqn(agent, env, n_episodes=2000, log_interval=100):
    """Train DQN agent."""
    episode_rewards = []
    episode_lengths = []
    best_avg_reward = -float('inf')
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=ep)
        total_reward = 0
        done = False
        steps = 0
        
        while not done:
            action, _ = agent.predict(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            agent.store(state, action, next_state, reward, float(done))
            agent.train_step()
            
            state = next_state
            total_reward += reward
            steps += 1
        
        episode_rewards.append(total_reward)
        episode_lengths.append(steps)
        
        if (ep + 1) % log_interval == 0:
            avg_r = np.mean(episode_rewards[-log_interval:])
            avg_l = np.mean(episode_lengths[-log_interval:])
            eps = agent.get_epsilon()
            print(f'Ep {ep+1:5d} | Avg reward: {avg_r:+8.3f} | Avg len: {avg_l:6.1f} | '
                  f'Eps: {eps:.4f} | Buffer: {len(agent.memory)}')
            
            if avg_r > best_avg_reward:
                best_avg_reward = avg_r
                torch.save(agent.policy_net.state_dict(), MODELS_DIR / 'dqn_best.pth')
    
    return episode_rewards, episode_lengths


# Train DQN
env = CyberDeceptionEnv(max_steps=100, atk_skill=0.5)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

dqn_agent = DQNAgent(obs_dim, n_actions, lr=3e-4, gamma=0.99,
                     eps_decay=3000, batch_size=64, target_update=300)

print(f'DQN training: obs_dim={obs_dim}, n_actions={n_actions}')
dqn_rewards, dqn_lengths = train_dqn(dqn_agent, env, n_episodes=2000, log_interval=200)

In [ ]:
# Plot DQN training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Smoothed rewards
window = 50
smoothed = np.convolve(dqn_rewards, np.ones(window)/window, mode='valid')
axes[0].plot(smoothed, color='#2196F3', linewidth=1.5)
axes[0].set_title('DQN: Episode Reward (smoothed)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].grid(True, alpha=0.3)

# Episode lengths
smoothed_l = np.convolve(dqn_lengths, np.ones(window)/window, mode='valid')
axes[1].plot(smoothed_l, color='#FF9800', linewidth=1.5)
axes[1].set_title('DQN: Episode Length (smoothed)')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Steps')
axes[1].grid(True, alpha=0.3)

# Loss
if dqn_agent.train_losses:
    loss_smooth = np.convolve(dqn_agent.train_losses, np.ones(100)/100, mode='valid')
    axes[2].plot(loss_smooth, color='#F44336', linewidth=1)
    axes[2].set_title('DQN: Training Loss (smoothed)')
    axes[2].set_xlabel('Step'); axes[2].set_ylabel('Loss')
    axes[2].grid(True, alpha=0.3)

plt.suptitle('DQN Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5. PPO Agent via Stable-Baselines3

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor


class RewardLoggingCallback(BaseCallback):
    """Log episode rewards during training."""
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
    
    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
                self.episode_lengths.append(info['episode']['l'])
        return True


# Wrap env
def make_env():
    def _init():
        e = CyberDeceptionEnv(max_steps=100, atk_skill=0.5)
        return Monitor(e)
    return _init

vec_env = DummyVecEnv([make_env()])

# PPO with custom policy size
ppo_model = PPO(
    'MlpPolicy',
    vec_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
        activation_fn=nn.ReLU,
    ),
    verbose=0,
    seed=SEED,
)

print(f'PPO model created. Policy: {ppo_model.policy}')

In [ ]:
# Train PPO
reward_callback = RewardLoggingCallback()

eval_env = DummyVecEnv([make_env()])
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(MODELS_DIR),
    eval_freq=5000,
    n_eval_episodes=20,
    deterministic=True,
    verbose=0,
)

total_timesteps = 200_000

print(f'Training PPO for {total_timesteps:,} timesteps...')
ppo_model.learn(
    total_timesteps=total_timesteps,
    callback=[reward_callback, eval_callback],
    progress_bar=True,
)

ppo_model.save(MODELS_DIR / 'ppo_deception')
print(f'\nPPO training complete. Model saved.')

In [ ]:
# Plot PPO training
if reward_callback.episode_rewards:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    window = 50
    r = np.array(reward_callback.episode_rewards)
    smoothed_r = np.convolve(r, np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed_r, color='#4CAF50', linewidth=1.5)
    axes[0].set_title('PPO: Episode Reward (smoothed)')
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
    axes[0].grid(True, alpha=0.3)
    
    l = np.array(reward_callback.episode_lengths)
    smoothed_l = np.convolve(l, np.ones(window)/window, mode='valid')
    axes[1].plot(smoothed_l, color='#FF9800', linewidth=1.5)
    axes[1].set_title('PPO: Episode Length (smoothed)')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Steps')
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('PPO Training Progress', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('No episode data logged (try increasing timesteps).')

---
## 6. Curriculum Learning — Progressive Difficulty

In [ ]:
def curriculum_training(model_class, make_env_fn, stages, total_per_stage=50_000):
    """
    Train with increasing attacker skill levels.
    This helps the agent learn basic defense first, then adapt to
    stronger adversaries.
    """
    results = []
    
    # Start fresh
    first_env = DummyVecEnv([lambda: Monitor(CyberDeceptionEnv(atk_skill=stages[0]))])
    model = PPO(
        'MlpPolicy', first_env,
        learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
        gamma=0.99, clip_range=0.2, ent_coef=0.01,
        policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
        verbose=0, seed=SEED,
    )
    
    for stage_idx, skill in enumerate(stages):
        print(f'\n--- Curriculum Stage {stage_idx+1}/{len(stages)}: attacker_skill={skill:.2f} ---')
        
        # Create env with current difficulty
        env = DummyVecEnv([lambda s=skill: Monitor(CyberDeceptionEnv(atk_skill=s))])
        model.set_env(env)
        
        cb = RewardLoggingCallback()
        model.learn(total_timesteps=total_per_stage, callback=cb, reset_num_timesteps=False)
        
        if cb.episode_rewards:
            avg_r = np.mean(cb.episode_rewards[-50:])
            print(f'  Final avg reward: {avg_r:.3f} (last 50 episodes)')
            results.append({'skill': skill, 'rewards': cb.episode_rewards})
    
    return model, results


# Curriculum stages: easy → medium → hard → expert
curriculum_stages = [0.2, 0.4, 0.6, 0.8]

print('Starting curriculum training...')
curriculum_model, curriculum_results = curriculum_training(
    PPO, make_env, curriculum_stages, total_per_stage=50_000
)

curriculum_model.save(MODELS_DIR / 'ppo_curriculum')
print('\nCurriculum model saved.')

In [ ]:
# Plot curriculum learning progress
if curriculum_results:
    fig, axes = plt.subplots(1, len(curriculum_results), figsize=(5*len(curriculum_results), 4))
    if len(curriculum_results) == 1: axes = [axes]
    
    colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
    for ax, res, color in zip(axes, curriculum_results, colors):
        r = np.array(res['rewards'])
        if len(r) > 20:
            sm = np.convolve(r, np.ones(20)/20, mode='valid')
            ax.plot(sm, color=color, linewidth=1.5)
        ax.set_title(f'Skill={res["skill"]:.1f}')
        ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Curriculum Learning Stages', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## 7. Evaluate All Agents

In [ ]:
def evaluate_all_agents(env, agents_dict, n_episodes=200):
    """Compare all agents on same environment."""
    results = {}
    
    for name, agent in agents_dict.items():
        ep_rewards, ep_hp_int, ep_real_comp, ep_det_time, ep_lens = [], [], [], [], []
        
        for ep in range(n_episodes):
            obs, _ = env.reset(seed=ep + 10000)
            total_reward = 0
            done = False
            steps = 0
            
            while not done:
                if hasattr(agent, 'predict'):
                    # SB3 or custom agent
                    if hasattr(agent, 'policy'):  # SB3
                        action, _ = agent.predict(obs, deterministic=True)
                    else:
                        action, _ = agent.predict(obs, deterministic=True) if hasattr(agent.predict, '__code__') and 'deterministic' in agent.predict.__code__.co_varnames else agent.predict(obs)
                else:
                    action = env.action_space.sample()
                
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += reward
                done = terminated or truncated
                steps += 1
            
            ep_rewards.append(total_reward)
            ep_hp_int.append(info['honeypot_interactions'])
            ep_real_comp.append(info['real_compromises'])
            ep_det_time.append(info['detection_time'])
            ep_lens.append(steps)
        
        results[name] = {
            'rewards': np.array(ep_rewards),
            'hp_interactions': np.array(ep_hp_int),
            'real_compromises': np.array(ep_real_comp),
            'detection_times': np.array(ep_det_time),
            'episode_lengths': np.array(ep_lens),
        }
        
        print(f'{name:20s} | Reward: {np.mean(ep_rewards):+7.3f} ± {np.std(ep_rewards):.3f} '
              f'| HP int: {np.mean(ep_hp_int):.2f} | Comp: {np.mean(ep_real_comp):.2f}')
    
    return results


# Build agent dict for comparison
class HeuristicDefender:
    def predict(self, obs, deterministic=False):
        det = obs[-11]>0.5; nc = obs[-5]; nhp = obs[-4]; np_ = obs[-3]
        if det and nc>0.2: return 8, None
        elif det and nhp>0: return 6, None
        elif np_>0.1 and nhp<0.5: return np.random.choice([1,2,3,4]), None
        elif np_>0: return 7, None
        return 0, None

eval_env = CyberDeceptionEnv(max_steps=100, atk_skill=0.5)

all_agents = {
    'Random': type('', (), {'predict': lambda self, obs, **kw: (eval_env.action_space.sample(), None)})(),
    'Heuristic': HeuristicDefender(),
    'DQN': dqn_agent,
    'PPO': ppo_model,
    'PPO-Curriculum': curriculum_model,
}

print('\nEvaluating all agents (200 episodes each)...\n')
all_results = evaluate_all_agents(eval_env, all_agents, n_episodes=200)

In [ ]:
# Comprehensive comparison plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Agent Performance Comparison', fontsize=16, fontweight='bold')

agent_names = list(all_results.keys())
colors = ['#9E9E9E', '#FF9800', '#2196F3', '#4CAF50', '#E91E63']

# Rewards distribution
ax = axes[0, 0]
data = [all_results[n]['rewards'] for n in agent_names]
bp = ax.boxplot(data, labels=agent_names, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_title('Episode Reward Distribution'); ax.grid(True, alpha=0.3)

# Honeypot interactions
ax = axes[0, 1]
means = [all_results[n]['hp_interactions'].mean() for n in agent_names]
stds = [all_results[n]['hp_interactions'].std() for n in agent_names]
bars = ax.bar(agent_names, means, yerr=stds, color=colors, alpha=0.7, capsize=5)
ax.set_title('Avg Honeypot Interactions'); ax.grid(True, alpha=0.3, axis='y')

# Real compromises
ax = axes[1, 0]
means = [all_results[n]['real_compromises'].mean() for n in agent_names]
stds = [all_results[n]['real_compromises'].std() for n in agent_names]
bars = ax.bar(agent_names, means, yerr=stds, color=colors, alpha=0.7, capsize=5)
ax.set_title('Avg Real Compromises (lower=better)'); ax.grid(True, alpha=0.3, axis='y')

# Detection timing
ax = axes[1, 1]
for i, name in enumerate(agent_names):
    dt = all_results[name]['detection_times']
    dt_valid = dt[dt > 0]
    if len(dt_valid) > 0:
        ax.hist(dt_valid, bins=20, alpha=0.5, label=name, color=colors[i])
ax.set_title('Detection Time Distribution'); ax.set_xlabel('Timestep')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 8. Export Best Agent & Utility Functions

In [ ]:
# Determine best agent
agent_scores = {name: res['rewards'].mean() for name, res in all_results.items()}
best_agent_name = max(agent_scores, key=agent_scores.get)
print(f'Best agent: {best_agent_name} (avg reward: {agent_scores[best_agent_name]:.3f})')

# Save DQN final
torch.save(dqn_agent.policy_net.state_dict(), MODELS_DIR / 'dqn_final.pth')

# Save evaluation results
eval_summary = {}
for name, res in all_results.items():
    eval_summary[name] = {
        'mean_reward': float(res['rewards'].mean()),
        'std_reward': float(res['rewards'].std()),
        'mean_hp_interactions': float(res['hp_interactions'].mean()),
        'mean_real_compromises': float(res['real_compromises'].mean()),
        'mean_episode_length': float(res['episode_lengths'].mean()),
    }

with open(MODELS_DIR / 'agent_comparison.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print(f'\nAll models saved to {MODELS_DIR}')
print('Files:')
for f in MODELS_DIR.glob('*'):
    print(f'  {f.name}')

In [ ]:
def load_deception_agent(model_type='ppo', model_path=None):
    """
    Load trained deception agent for inference.
    Used by Notebook 09 (Orchestration) and 10 (API).
    """
    if model_path is None:
        model_path = MODELS_DIR
    else:
        model_path = Path(model_path)
    
    if model_type == 'ppo':
        env = DummyVecEnv([lambda: CyberDeceptionEnv()])
        model = PPO.load(model_path / 'ppo_deception', env=env)
        return model
    
    elif model_type == 'ppo_curriculum':
        env = DummyVecEnv([lambda: CyberDeceptionEnv()])
        model = PPO.load(model_path / 'ppo_curriculum', env=env)
        return model
    
    elif model_type == 'dqn':
        obs_dim = 95  # 12*7 + 11
        n_actions = 9
        net = DuelingDQN(obs_dim, n_actions).to(DEVICE)
        net.load_state_dict(torch.load(model_path / 'dqn_final.pth', map_location=DEVICE))
        net.eval()
        
        class DQNWrapper:
            def __init__(self, net):
                self.net = net
            def predict(self, obs, deterministic=True):
                with torch.no_grad():
                    t = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
                    return self.net(t).argmax(1).item(), None
        
        return DQNWrapper(net)


# Test load
loaded_agent = load_deception_agent('ppo')
test_obs, _ = CyberDeceptionEnv().reset()
action, _ = loaded_agent.predict(test_obs, deterministic=True)
print(f'Loaded PPO agent. Test action: {action}')
print('\n✓ Notebook 06 complete. Ready for Notebook 07 (Threat Intelligence Memory).')